# CMFE v3 — Stain-robust LoRA adaptation of a pathology foundation model (BreakHis, patient-level, + BACH external)

**Before you run (Kaggle):**
1. Settings → Accelerator → **GPU T4**, Settings → **Internet ON**.
2. **Add Input → search "BACH" → add `truthisneverlinear/bach-breast-cancer-histology-images`**.
   Kaggle blocks downloading new datasets during "Save & Run All", so the external dataset must be
   attached in the editor. BreakHis (`ambarish/breakhis`) can be attached the same way.
3. Set `QUICK_TEST = True` for a ~40-min smoke test. If it finishes, set it to `False` and do the full
   run (about 7–8 h on one T4; switch blocks off in `CFG` if you need it shorter).

## Why v3 is designed this way (lessons from v1 and v2)
- In v1 and v2 every method, including CMFE, sat on the **same frozen features**. All heads tied
  (binary ~89 %, subtype ~43 % balanced accuracy) because the ceiling is set by the frozen features,
  not by the head. The GRL, gating, PD-SupCon and the stain-feature bank were all non-significant.
- Published evidence: for pathology foundation models, **LoRA adaptation beats linear probing in 83 %
  of in-domain comparisons** (Benchmarking Pathology Foundation Models, 2024). Full fine-tuning is
  less reliable on small data. Our own end-to-end ResNet-50 was the worst model, which agrees.
- The pathology model (Phikon-v2) beat the general model (DINOv2) by about 5 pp. So we adapt the
  pathology model.

## Proposed model (fixed before any v3 result was seen)
**Ours = Phikon-v2 (ViT-L) + LoRA (rank 8 on Q/K/V of every block) + on-the-fly H&E stain augmentation
(HED jitter, Tellez 2019) + geometric augmentation + class-balanced cross-entropy + 5-view test-time
averaging.** About 0.8 M trainable parameters (0.3 % of the backbone). Adaptation now happens inside
the backbone, so the model can move past the frozen-feature ceiling.

**Baselines (same patient-disjoint folds):** a linear probe on each frozen foundation model, LogReg,
RBF-SVM, k-NN and an MLP on the concatenated frozen features, and end-to-end fine-tuned ResNet-50.

**Ablations (subtype task):** no stain augmentation; + patient-decorrelated SupCon; LoRA on DINOv2
instead of Phikon-v2.

**Evidence produced:** patient-cluster bootstrap 95 % CIs for every comparison, cross-patient retrieval,
the leakage demonstration, BACH external validation, and an automatic claims checklist.
**Write only SUPPORTED claims into the paper.**

## 1. Environment and configuration

In [1]:
import os, sys, re, json, time, math, random, hashlib, subprocess, importlib, warnings, shutil, copy
from pathlib import Path
from collections import defaultdict

DRY_RUN = os.environ.get('CMFE_DRY_RUN', '0') == '1'   # leave as is on Kaggle (real run)

# >>> QUICK_TEST = True: ~40-min smoke test on the REAL data (image subset, 1 epoch).
# >>> When it finishes without errors, set it to False for the paper run.
QUICK_TEST = False or os.environ.get('CMFE_QUICK', '0') == '1'


def _ensure(pkgs):
    for pip_name, mod in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            print(f'installing {pip_name} ...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

_ensure([('timm', 'timm'), ('transformers', 'transformers'), ('scikit-image', 'skimage')]
        + ([] if DRY_RUN else [('kagglehub', 'kagglehub')]))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import timm
from skimage.color import rgb2hed, hed2rgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

WORK = Path(os.environ.get('CMFE_WORK', '/kaggle/working'))
FEAT_DIR, RES_DIR, FIG_DIR = WORK / 'features', WORK / 'results', WORK / 'figures'
for d in (FEAT_DIR, RES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    dataset_slug='ambarish/breakhis',
    external_slug='truthisneverlinear/bach-breast-cancer-histology-images',
    backbones=['dinov2_vitb14', 'phikon_v2'],      # frozen baselines; optional: 'hoptimus0', 'uni2' (HF_TOKEN)
    n_folds=5, tasks=['binary', 'subtype'],
    batch_extract=32, num_workers=4,
    # ---- proposed model (LoRA) ----
    lora_backbone='phikon_v2', lora_r=8, lora_alpha=16, lora_dropout=0.1,
    lora_epochs=6, lora_patience=3, lora_bs=16, lora_lr=2e-4, head_lr=1e-3, lora_wd=1e-4,
    lora_seeds=[0], label_smooth=0.1, supcon_lam=0.2, tau=0.1,
    # ---- frozen-feature MLP head baseline ----
    hid=256, emb=128, epochs=80, patience=15, lr=1e-3, wd=1e-4, batch=256, p_in_drop=0.2,
    # ---- other baselines ----
    logreg_C=[0.01, 0.1, 1.0], svm_C=10.0, svm_pca=256, knn_k=10,
    run_finetune=True, ft_epochs=10, ft_lr=3e-4, ft_bs=48,
    # ---- blocks ----
    run_ablation=True, abl_supcon=False,          # PD-SupCon ablation doubles cost; v2 showed no benefit
    run_leakage_demo=True, run_external=True,
    n_boot=2000,
)
if QUICK_TEST and not DRY_RUN:
    CFG.update(lora_epochs=1, epochs=20, patience=5, ft_epochs=1, n_boot=200)
if DRY_RUN:
    CFG.update(n_folds=2, lora_epochs=2, lora_patience=2, lora_bs=8, epochs=30, patience=10, ft_epochs=1,
               n_boot=50, batch_extract=8, num_workers=0, svm_pca=16, batch=64, logreg_C=[0.1])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)

set_seed(42)
print(f'DRY_RUN={DRY_RUN} | QUICK_TEST={QUICK_TEST} | device={DEVICE} | torch {torch.__version__} | timm {timm.__version__}')
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
elif not DRY_RUN:
    print('WARNING: no GPU — Settings -> Accelerator -> GPU')

if not DRY_RUN:   # optional HF token for gated models (Add-ons -> Secrets -> HF_TOKEN)
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')
        if tok:
            from huggingface_hub import login
            login(token=tok); print('HF token loaded')
    except Exception:
        pass
json.dump(CFG, open(RES_DIR / 'config.json', 'w'), indent=2)

DRY_RUN=False | QUICK_TEST=False | device=cuda | torch 2.11.0+cu128 | timm 1.0.29
GPU: Tesla T4


## 2a. Dataset download (Kaggle)

In [2]:
if not DRY_RUN:
    import kagglehub
    path = kagglehub.dataset_download("ambarish/breakhis")
    print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/ambarish/breakhis


## 2b. BreakHis metadata (82 patients, 4 magnifications, 8 subtypes)

In [3]:
SUBTYPES = ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
MAGS = [40, 100, 200, 400]
RX = re.compile(r'SOB_(?P<cls>[BM])_(?P<sub>[A-Z]+)-(?P<year>\d+)-(?P<pid>[0-9A-Za-z]+)-'
                r'(?P<mag>\d+)-(?P<seq>\d+)\.png$', re.I)


def _synth_img(rng, si, mag, h=460, w=700):
    base = np.array([200 - 8 * si, 120 + 5 * si, 190 - 10 * si], dtype=np.float32)   # pink/purple, H&E-like
    img = base + rng.normal(0, 25, (h, w, 3))
    yy, xx = np.mgrid[0:h, 0:w]
    img[..., 1] += 30 * np.sin(xx / (5 + si) + mag / 100.0)
    return Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))


def make_synthetic(root, n_pat_per_sub=2, n_img=3):
    """Tiny BreakHis-shaped dataset (for testing the code only)."""
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    rng = np.random.default_rng(0)
    pid = 1000
    for si, sub in enumerate(SUBTYPES):
        cls = 'B' if si < 4 else 'M'
        for _ in range(n_pat_per_sub):
            pid += 1
            for mag in MAGS:
                d = root / 'histology_slides' / 'breast' / sub / f'SOB_{cls}_{sub}_14-{pid}' / f'{mag}X'
                d.mkdir(parents=True, exist_ok=True)
                for k in range(1, n_img + 1):
                    _synth_img(rng, si, mag).save(d / f'SOB_{cls}_{sub}-14-{pid}-{mag}-{k:03d}.png')
    return root


DATA_ROOT = make_synthetic(WORK / 'synthetic_breakhis') if DRY_RUN else Path(path)
rows, n_bad, seen = [], 0, set()
for p in sorted(DATA_ROOT.rglob('*')):
    if p.suffix.lower() != '.png':
        continue
    m = RX.search(p.name)
    if not m:
        n_bad += 1
        continue
    if p.name in seen:
        continue
    seen.add(p.name)
    g = m.groupdict()
    rows.append(dict(path=str(p), file=p.name, subtype=g['sub'].upper(), binary=int(g['cls'].upper() == 'M'),
                     patient=f"{g['year']}-{g['pid']}", mag=int(g['mag'])))
df = pd.DataFrame(rows)
assert len(df), f'No BreakHis PNGs found under {DATA_ROOT}.'
df = df[df.subtype.isin(SUBTYPES) & df.mag.isin(MAGS)].reset_index(drop=True)
print(f'images: {len(df)} | patients: {df.patient.nunique()} | unparsed png: {n_bad}')
if QUICK_TEST:
    df = df.groupby(['patient', 'mag']).head(4).reset_index(drop=True)
    print('QUICK_TEST subset:', len(df), 'images')
print(pd.crosstab(df.subtype, df.mag, margins=True))

images: 7909 | patients: 81 | unparsed png: 0
mag        40   100   200   400   All
subtype                              
A         114   113   111   106   444
DC        864   903   896   788  3451
F         253   260   264   237  1014
LC        156   170   163   137   626
MC        205   222   196   169   792
PC        145   142   135   138   560
PT        109   121   108   115   453
TA        149   150   140   130   569
All      1995  2081  2013  1820  7909


## 3. Integrity audit (exact duplicates, before any split)

In [4]:
def pixel_hash(p):
    with Image.open(p) as im:
        return hashlib.md5(np.asarray(im.convert('RGB')).tobytes()).hexdigest()

t0 = time.time()
df['hash'] = [pixel_hash(p) for p in df.path]
drop, audit = set(), []
for h, g in df.groupby('hash'):
    if len(g) < 2:
        continue
    kind = ('cross_label' if g.subtype.nunique() > 1 else
            'cross_patient' if g.patient.nunique() > 1 else 'within_patient')
    audit.append(dict(hash=h, kind=kind, n=len(g), patients='|'.join(sorted(set(g.patient))),
                      subtypes='|'.join(sorted(set(g.subtype))), files='|'.join(g.file)))
    drop |= set(g.index) if kind == 'cross_label' else set(g.index[1:])
audit = pd.DataFrame(audit, columns=['hash', 'kind', 'n', 'patients', 'subtypes', 'files'])
audit.to_csv(RES_DIR / 'integrity_audit.csv', index=False)
print(f'hashed in {time.time()-t0:.0f}s | duplicate groups: {len(audit)}',
      dict(audit.kind.value_counts()) if len(audit) else '')
if len(audit):
    print(audit.groupby(['kind', 'patients', 'subtypes']).size().rename('groups').to_string())
df = df.drop(index=sorted(drop)).reset_index(drop=True)
print(f'after cleanup: {len(df)} images')

df['y_binary'] = df.binary.values
df['y_subtype'] = df.subtype.map({s: i for i, s in enumerate(SUBTYPES)}).values
df['mag_id'] = df.mag.map({m: i for i, m in enumerate(MAGS)}).values
N = len(df)
TASK_C = {'binary': 2, 'subtype': len(SUBTYPES)}
TASK_NAMES = {'binary': ['Benign', 'Malignant'], 'subtype': SUBTYPES}

hashed in 179s | duplicate groups: 125 {'cross_label': np.int64(123), 'within_patient': np.int64(2)}
kind            patients  subtypes
cross_label     14-13412  DC|LC       123
within_patient  14-12204  LC            2
after cleanup: 7661 images


## 4. Patient-disjoint folds (identical to v1/v2: same seed, same cleanup)

In [5]:
pat_label = df.groupby('patient').y_subtype.agg(lambda s: s.mode()[0])
df['strat'] = df.patient.map(pat_label)
sgkf = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=42)
df['fold'] = -1
for f, (_, te) in enumerate(sgkf.split(df, df.strat, groups=df.patient)):
    df.loc[te, 'fold'] = f
assert (df.groupby('patient').fold.nunique() == 1).all(), 'patient leakage across folds'
print(pd.crosstab(df.fold, df.subtype))
print('patients per fold:', df.groupby('fold').patient.nunique().to_dict())
df.drop(columns=['hash']).to_csv(RES_DIR / 'metadata_folds.csv', index=False)

PATIENT = df.patient.values
PID_INT = pd.factorize(PATIENT)[0]
MAG = df.mag_id.values
PATHS = df.path.values


def inner_split(tr_idx, seed, groups=None):
    g = PATIENT[tr_idx] if groups is None else groups[tr_idx]
    strat = df.strat.values[tr_idx]
    k = 5
    while k >= 2:
        try:
            spl = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
            a, b = next(spl.split(tr_idx, strat, groups=g))
            return tr_idx[a], tr_idx[b]
        except ValueError:
            k -= 1
    raise RuntimeError('inner split failed')

subtype    A    DC    F   LC   MC   PC   PT   TA
fold                                            
0        121  1129    0  201    0   74    0  124
1          0   300  399    0  179  152  218  129
2        193   506  247   74   90    0  235  197
3          0   577  162  226  459  269    0    0
4        130   816  206    0   64   65    0  119
patients per fold: {0: 17, 1: 15, 2: 15, 3: 16, 4: 17}


## 5. Backbones, augmentation, frozen-feature extraction (for the baselines)

In [6]:
IMNET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))


def load_backbone(name):
    """Returns (forward_fn, mean, std, model). forward_fn: Bx3x224x224 normalised -> BxD."""
    if name == 'dinov2_vitb14':
        arch = 'vit_small_patch14_dinov2' if DRY_RUN else 'vit_base_patch14_dinov2.lvd142m'
        m = timm.create_model(arch, pretrained=not DRY_RUN, num_classes=0, img_size=224)
        return (lambda x: m(x)), IMNET[0], IMNET[1], m
    if name == 'phikon_v2':
        from transformers import AutoModel, AutoImageProcessor, Dinov2Config, Dinov2Model
        if DRY_RUN:
            m = Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                                         intermediate_size=128, patch_size=16, image_size=224))
            mean, std = IMNET
        else:
            try:
                m = AutoModel.from_pretrained('owkin/phikon-v2', attn_implementation='sdpa')   # memory-efficient attention
            except Exception:
                m = AutoModel.from_pretrained('owkin/phikon-v2')
            proc = AutoImageProcessor.from_pretrained('owkin/phikon-v2')
            mean, std = tuple(proc.image_mean), tuple(proc.image_std)
        return (lambda x: m(pixel_values=x).last_hidden_state[:, 0]), mean, std, m
    if name == 'hoptimus0':
        m = timm.create_model('hf-hub:bioptimus/H-optimus-0', pretrained=True, init_values=1e-5,
                              dynamic_img_size=False)
        return (lambda x: m(x)), (0.707223, 0.578729, 0.703617), (0.211883, 0.230117, 0.177517), m
    if name == 'uni2':
        m = timm.create_model('hf-hub:MahmoodLab/UNI2-h', pretrained=True, img_size=224, patch_size=14,
                              depth=24, num_heads=24, init_values=1e-5, embed_dim=1536, mlp_ratio=2.66667 * 2,
                              num_classes=0, no_embed_class=True, mlp_layer=timm.layers.SwiGLUPacked,
                              act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True)
        return (lambda x: m(x)), IMNET[0], IMNET[1], m
    raise ValueError(name)


def hed_jitter(img, rng, sigma=0.05):
    """H&E stain augmentation in HED space (Tellez et al. 2019), applied as a delta (zero jitter = identity)."""
    a = np.asarray(img).astype(np.float32) / 255.0
    hed = rgb2hed(a)
    alpha = rng.uniform(1 - sigma, 1 + sigma, 3)
    beta = rng.uniform(-sigma, sigma, 3)
    hed_j = hed * alpha + beta * np.abs(hed).max(axis=(0, 1), keepdims=True)
    rgb = np.clip(a + hed2rgb(hed_j) - hed2rgb(hed), 0, 1)
    return Image.fromarray((rgb * 255).astype(np.uint8))


def five_views(img):
    g = TF.to_tensor(img.resize((224, 224), Image.BICUBIC))
    big = TF.to_tensor(img.resize((448, 448), Image.BICUBIC))
    return torch.stack([g] + [big[:, r:r + 224, c:c + 224] for r in (0, 224) for c in (0, 224)])


class ViewDS(Dataset):
    """5 deterministic views per image (global + 4 tiles) for evaluation and feature extraction."""
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return five_views(Image.open(self.paths[i]).convert('RGB'))


class TrainDS(Dataset):
    """Random crop + rot90/flip (+ HED stain + brightness/contrast jitter). Optionally two views per image."""
    def __init__(self, paths, y, stain=True, two_views=False):
        self.paths, self.y, self.stain, self.two = list(paths), np.asarray(y), stain, two_views
        self.rrc = T.RandomResizedCrop(224, scale=(0.25, 1.0), ratio=(0.75, 1.33))

    def __len__(self):
        return len(self.paths)

    def _aug(self, img, rng):
        img = self.rrc(img)
        img = img.rotate(90 * int(rng.integers(0, 4)))
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if self.stain:
            img = hed_jitter(img, rng)
            img = TF.adjust_brightness(img, float(rng.uniform(0.9, 1.1)))
            img = TF.adjust_contrast(img, float(rng.uniform(0.9, 1.1)))
        return TF.to_tensor(img)

    def __getitem__(self, i):
        rng = np.random.default_rng()
        img = Image.open(self.paths[i]).convert('RGB')
        x = self._aug(img, rng)
        if self.two:
            x = torch.stack([x, self._aug(img, rng)])
        return x, int(self.y[i]), i


@torch.no_grad()
def extract_frozen(name, paths, tag):
    cache = FEAT_DIR / f'{name}_{tag}.npz'
    if cache.exists():
        d = np.load(cache, allow_pickle=True)
        if list(d['paths']) == list(paths):
            print(f'[{name}/{tag}] cached {d["X"].shape}')
            return d['X']
    fn, mean, std, model = load_backbone(name)
    model.eval().to(DEVICE)
    mean = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    dl = DataLoader(ViewDS(paths), batch_size=CFG['batch_extract'], num_workers=CFG['num_workers'], pin_memory=AMP)
    feats, t0 = [], time.time()
    for bi, x in enumerate(dl):
        B = x.shape[0]
        x = (x.view(B * 5, 3, 224, 224).to(DEVICE, non_blocking=True) - mean) / std
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            f = fn(x).float()
        f = f.view(B, 5, -1)
        feats.append(torch.cat([f[:, 0], f[:, 1:].mean(1)], 1).cpu().numpy())
        if bi % 60 == 0:
            print(f'  [{name}/{tag}] batch {bi+1}/{len(dl)}  {time.time()-t0:.0f}s')
    X = np.concatenate(feats).astype(np.float32)
    np.savez(cache, X=X, paths=np.array(list(paths)))
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print(f'[{name}/{tag}] {X.shape} in {time.time()-t0:.0f}s')
    return X


TAG = 'dry' if DRY_RUN else ('quick' if QUICK_TEST else 'full')
FEATS, BB = [], []
for name in CFG['backbones']:
    try:
        FEATS.append(extract_frozen(name, PATHS, TAG)); BB.append(name)
    except Exception as e:
        print(f'!! backbone {name} failed and is skipped: {type(e).__name__}: {e}')
assert BB, 'no backbone could be loaded (is Internet ON?)'
print('frozen backbones:', BB, '| dims:', [x.shape[1] for x in FEATS])

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

  [dinov2_vitb14/full] batch 1/240  5s
  [dinov2_vitb14/full] batch 61/240  37s
  [dinov2_vitb14/full] batch 121/240  74s
  [dinov2_vitb14/full] batch 181/240  117s
[dinov2_vitb14/full] (7661, 1536) in 156s


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

  [phikon_v2/full] batch 1/240  7s
  [phikon_v2/full] batch 61/240  102s
  [phikon_v2/full] batch 121/240  197s
  [phikon_v2/full] batch 181/240  291s
[phikon_v2/full] (7661, 2048) in 383s
frozen backbones: ['dinov2_vitb14', 'phikon_v2'] | dims: [1536, 2048]


## 6. Proposed model — LoRA-adapted pathology foundation model

In [7]:
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, B initialised to zero (starts identical to the base)."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Linear(base.in_features, r, bias=False)
        self.B = nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.drop(x))) * self.scale


def add_lora(model, r, alpha, dropout, targets=('query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj', 'qkv')):
    """Wraps attention Q/K/V projections (HF old: query/key/value; HF new: q_proj/k_proj/v_proj; timm: qkv)."""
    n = 0
    for module in list(model.modules()):
        for cname, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(module, cname, LoRALinear(child, r, alpha, dropout)); n += 1
    return n


class PeftNet(nn.Module):
    def __init__(self, backbone, n_cls):
        super().__init__()
        fn, mean, std, m = load_backbone(backbone)
        for p in m.parameters():
            p.requires_grad = False
        self.n_lora = add_lora(m, CFG['lora_r'], CFG['lora_alpha'], CFG['lora_dropout'])
        assert self.n_lora > 0, f'LoRA found no attention projections in {backbone} — check layer names'
        self.m, self.fn = m, fn
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
        with torch.no_grad():
            D = fn(torch.zeros(1, 3, 224, 224)).shape[-1]
        self.head = nn.Sequential(nn.LayerNorm(D), nn.Dropout(0.1), nn.Linear(D, n_cls))
        self.proj = nn.Sequential(nn.Linear(D, 256), nn.GELU(), nn.Linear(256, 128))

    def features(self, x):
        return self.fn((x - self.mean) / self.std)

    def forward(self, x):
        f = self.features(x)
        return self.head(f), F.normalize(self.proj(f.float()), dim=1), f


def supcon_loss(z, y, pid, inst, tau):
    """Patient-decorrelated multi-view SupCon: positives = other view of same image or same class from a
    different patient; other images of the same patient are excluded from positives and denominator."""
    B = z.shape[0]
    sim = z @ z.T / tau
    valid = ~torch.eye(B, dtype=torch.bool, device=z.device)
    valid &= (pid[:, None] != pid[None, :]) | (inst[:, None] == inst[None, :])
    pos = (y[:, None] == y[None, :]) & valid
    sim = sim.masked_fill(~valid, -1e4)
    logp = sim - torch.logsumexp(sim, 1, keepdim=True)
    npos = pos.sum(1)
    ok = npos > 0
    if not ok.any():
        return z.sum() * 0
    return (-(logp * pos).sum(1)[ok] / npos[ok]).mean()


@torch.no_grad()
def peft_predict(model, paths):
    """5-view test-time averaging. Returns logits (N,C) and L2-normalised CLS features (N,D)."""
    model.eval()
    dl = DataLoader(ViewDS(paths), batch_size=max(1, CFG['lora_bs'] // 2), num_workers=CFG['num_workers'], pin_memory=AMP)
    L, Fz = [], []
    for x in dl:
        B = x.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            lg, _, f = model(x.view(B * 5, 3, 224, 224).to(DEVICE, non_blocking=True))
        L.append(lg.float().view(B, 5, -1).mean(1).cpu()); Fz.append(F.normalize(f.float().view(B, 5, -1).mean(1), dim=1).cpu())
    return torch.cat(L).numpy(), torch.cat(Fz).numpy()


def trainable_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}


def train_peft(tr, va, y, C, seed, backbone=None, stain=True, supcon=False, epochs=None):
    set_seed(seed)
    model = PeftNet(backbone or CFG['lora_backbone'], C).to(DEVICE)
    lora_p = [p for n, p in model.named_parameters() if p.requires_grad and ('.A.' in n or '.B.' in n)]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not ('.A.' in n or '.B.' in n)]
    n_tr = sum(p.numel() for p in lora_p + head_p)
    n_all = sum(p.numel() for p in model.m.parameters())
    print(f'    LoRA modules {model.n_lora} | trainable {n_tr/1e6:.2f}M / backbone {n_all/1e6:.1f}M')
    opt = torch.optim.AdamW([{'params': lora_p, 'lr': CFG['lora_lr']}, {'params': head_p, 'lr': CFG['head_lr']}],
                            weight_decay=CFG['lora_wd'])
    ds = TrainDS(PATHS[tr], y[tr], stain=stain, two_views=supcon)
    bs = max(2, CFG['lora_bs'] // 2) if supcon else CFG['lora_bs']      # two views per image -> half the images
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=CFG['num_workers'], pin_memory=AMP,
                    drop_last=len(tr) > bs, persistent_workers=CFG['num_workers'] > 0)
    E = epochs or CFG['lora_epochs']
    total = max(2, E * len(dl)); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    cnt = np.bincount(y[tr], minlength=C).astype(np.float32)
    cw = np.where(cnt > 0, cnt.sum() / (C * np.maximum(cnt, 1)), 0.0)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(cw, device=DEVICE), label_smoothing=CFG['label_smooth'])
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    pid_all = torch.as_tensor(PID_INT, device=DEVICE)
    keep_va = va[np.isin(y[va], np.unique(y[tr]))]
    best, best_state, bad = (-1, 1e9), None, 0
    for ep in range(E):
        model.train(); t0 = time.time()
        for x, yb, ib in dl:
            yb = yb.to(DEVICE)
            if supcon:
                B = x.shape[0]
                x = x.transpose(0, 1).reshape(2 * B, 3, 224, 224)
                yb2, pb = yb.repeat(2), pid_all[torch.as_tensor(tr, device=DEVICE)[ib.to(DEVICE)]].repeat(2)
                inst = torch.arange(B, device=DEVICE).repeat(2)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                lg, z, _ = model(x.to(DEVICE, non_blocking=True))
                if supcon:
                    loss = ce(lg.float(), yb2) + CFG['supcon_lam'] * supcon_loss(z.float(), yb2, pb, inst, CFG['tau'])
                else:
                    loss = ce(lg.float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        if len(keep_va) == 0:                       # no usable validation: fixed epochs, keep the last state
            best_state = trainable_state(model)
            print(f'    epoch {ep+1}/{E}: (no usable validation images)')
            continue
        lv, _ = peft_predict(model, PATHS[keep_va])
        score = (balanced_accuracy_score(y[keep_va], lv.argmax(1)),
                 F.cross_entropy(torch.tensor(lv), torch.tensor(y[keep_va])).item())
        print(f'    epoch {ep+1}/{E}: val bal_acc {score[0]:.4f}  val nll {score[1]:.4f}  ({time.time()-t0:.0f}s)')
        if score[0] > best[0] + 1e-6 or (abs(score[0] - best[0]) <= 1e-6 and score[1] < best[1]):
            best, best_state, bad = score, trainable_state(model), 0
        else:
            bad += 1
            if bad >= CFG['lora_patience']:
                break
    model.load_state_dict(best_state, strict=False)
    return model


def free(model):
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

## 7. Frozen-feature baselines, metrics, significance helpers

In [8]:
class MLPHead(nn.Module):
    def __init__(self, D, C, hid, emb, p):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(D, hid), nn.LayerNorm(hid), nn.GELU(), nn.Linear(hid, emb))
        self.out = nn.Linear(emb, C)

    def forward(self, x):
        z = self.net(x)
        return self.out(F.gelu(z)), F.normalize(z, dim=1)


def train_mlp(Xtr, ytr, Xva, yva, C, seed):
    set_seed(seed)
    model = MLPHead(Xtr.shape[1], C, CFG['hid'], CFG['emb'], CFG['p_in_drop']).to(DEVICE)
    Xt, yt = torch.tensor(Xtr, device=DEVICE), torch.tensor(ytr, device=DEVICE)
    Xv = torch.tensor(Xva, device=DEVICE)
    cnt = np.bincount(ytr, minlength=C).astype(np.float32)
    cw = np.where(cnt > 0, cnt.sum() / (C * np.maximum(cnt, 1)), 0.0)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(cw, device=DEVICE), label_smoothing=CFG['label_smooth'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    n, bs = len(ytr), min(CFG['batch'], len(ytr))
    keep = np.isin(yva, np.unique(ytr))
    best, best_state, bad = -1, None, 0
    for ep in range(CFG['epochs']):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for s in range(max(1, n // bs)):
            b = perm[s * bs:(s + 1) * bs]
            loss = ce(model(Xt[b])[0], yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xv)[0].argmax(1).cpu().numpy()
        sc = balanced_accuracy_score(yva[keep], pv[keep]) if keep.any() else 0
        if sc > best + 1e-6:
            best, best_state, bad = sc, copy.deepcopy(model.state_dict()), 0
        elif ep >= 10:
            bad += 1
            if bad >= CFG['patience']:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model


def mlp_out(model, X):
    with torch.no_grad():
        lg, z = model(torch.tensor(X, device=DEVICE))
    return lg.cpu().numpy(), z.cpu().numpy()


def logreg_fit(Xtr, ytr, Xva, yva, C):
    best = None
    for c in CFG['logreg_C']:
        clf = LogisticRegression(C=c, max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
        s = balanced_accuracy_score(yva, full_proba(clf, Xva, C).argmax(1))
        if best is None or s > best[0]:
            best = (s, clf)
    return best[1]


def full_proba(clf, X, C):
    P = np.zeros((len(X), C), dtype=np.float64)
    P[:, clf.classes_] = clf.predict_proba(X)
    return P


def softmax_np(l):
    l = l - l.max(1, keepdims=True)
    e = np.exp(l)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits_va, y_va):
    best_T, best = 1.0, 1e18
    for Tt in np.exp(np.linspace(-2.5, 2.5, 101)):
        p = softmax_np(logits_va / Tt)
        nll = -np.log(p[np.arange(len(y_va)), y_va] + 1e-12).mean()
        if nll < best:
            best, best_T = nll, Tt
    return best_T


def ece_score(P, y, n_bins=15):
    conf, pred = P.max(1), P.argmax(1)
    e = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e


def macro_auc(y, P):
    C = P.shape[1]
    if C == 2:
        return roc_auc_score(y, P[:, 1]) if len(np.unique(y)) == 2 else np.nan
    a = [roc_auc_score(y == c, P[:, c]) for c in range(C) if 0 < (y == c).sum() < len(y)]
    return float(np.mean(a)) if a else np.nan


def patient_agg(P, y, pids):
    C = P.shape[1]
    d = pd.DataFrame(P).assign(pid=pids, y=y)
    g = d.groupby('pid')
    Pp = g[list(range(C))].mean().values
    yp = g.y.agg(lambda s: s.mode()[0]).values
    corr = pd.Series(P.argmax(1) == y).groupby(pids).mean()
    return Pp, yp, float(corr.mean())


def all_metrics(P, y, pids, P_raw=None):
    pred = P.argmax(1)
    Pp, yp, prr = patient_agg(P, y, pids)
    out = dict(acc=accuracy_score(y, pred), bal_acc=balanced_accuracy_score(y, pred),
               macro_f1=f1_score(y, pred, average='macro'), auc=macro_auc(y, P), ece=ece_score(P, y),
               pat_acc=accuracy_score(yp, Pp.argmax(1)), pat_bal_acc=balanced_accuracy_score(yp, Pp.argmax(1)), prr=prr)
    if P_raw is not None:
        out['ece_raw'] = ece_score(P_raw, y)
    return out


ROWS = []


def record(exp, task, method, fold, seed, P_te, y_te, pid_te, logit_va=None, y_va=None, logit_te=None, extra=None):
    """logit_* are logits or log-probs; the temperature is fit on validation only."""
    P_raw = P_te
    if logit_va is not None and logit_te is not None:
        P_te = softmax_np(logit_te / fit_temperature(logit_va, y_va))
    ROWS.append(dict(experiment=exp, task=task, method=method, fold=fold, seed=seed,
                     **all_metrics(P_te, y_te, pid_te, P_raw), **(extra or {})))
    return P_te


def retrieval_ap(E, y, pid, k=10):
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    S = E @ E.T
    ap = np.full(len(E), np.nan); pk = np.full(len(E), np.nan)
    for i in range(len(E)):
        gi = np.where(pid != pid[i])[0]
        if len(gi) == 0:
            continue
        rel = (y[gi] == y[i])[np.argsort(-S[i, gi])]
        R = int(rel.sum())
        if R == 0:
            continue
        pk[i] = rel[:k].mean()
        ap[i] = (np.cumsum(rel[:R]) / np.arange(1, R + 1) * rel[:R]).sum() / R
    return ap, pk


AP_OOF = defaultdict(dict)


def add_retrieval(task, method, f, E, te, y):
    ap, pk = retrieval_ap(E, y[te], PATIENT[te])
    ROWS.append(dict(experiment='retrieval', task=task, method=method, fold=f, seed=0,
                     p_at_10=np.nanmean(pk), map_at_r=np.nanmean(ap)))
    AP_OOF[task].setdefault(method, np.full(N, np.nan))[te] = ap


def scale(tr, X):
    sc = StandardScaler().fit(X[tr])
    return sc.transform(X).astype(np.float32), sc


class FTDS(Dataset):
    def __init__(self, paths, y, train):
        self.p, self.y = list(paths), list(y)
        norm = T.Normalize(*IMNET)
        self.tf = (T.Compose([T.Resize((256, 256)), T.RandomResizedCrop(224, scale=(0.6, 1.0)),
                              T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                              T.ColorJitter(0.1, 0.1, 0.1, 0.02), T.ToTensor(), norm]) if train else
                   T.Compose([T.Resize((224, 224)), T.ToTensor(), norm]))

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]


def finetune_resnet50(tr, va, te, y, C, seed=0):
    set_seed(seed)
    model = timm.create_model('resnet18' if DRY_RUN else 'resnet50.a1_in1k', pretrained=not DRY_RUN, num_classes=C).to(DEVICE)
    mk = lambda idx, trn: DataLoader(FTDS(PATHS[idx], y[idx], trn), batch_size=CFG['ft_bs'], shuffle=trn,
                                     num_workers=CFG['num_workers'], pin_memory=AMP, drop_last=trn and len(idx) > CFG['ft_bs'])
    dl_tr, dl_va, dl_te = mk(tr, True), mk(va, False), mk(te, False)
    cnt = np.bincount(y[tr], minlength=C).astype(np.float32)
    cw = np.where(cnt > 0, cnt.sum() / (C * np.maximum(cnt, 1)), 0.0)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, device=DEVICE), label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['ft_lr'], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG['ft_epochs'] * max(1, len(dl_tr)))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)

    @torch.no_grad()
    def logits_of(dl):
        model.eval(); out = []
        for x, _ in dl:
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                out.append(model(x.to(DEVICE)).float().cpu())
        return torch.cat(out).numpy()

    best, best_state = -1, None
    for ep in range(CFG['ft_epochs']):
        model.train()
        for x, yy in dl_tr:
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                loss = crit(model(x.to(DEVICE)), yy.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        s = balanced_accuracy_score(y[va], logits_of(dl_va).argmax(1))
        if s > best:
            best, best_state = s, copy.deepcopy(model.state_dict())
    print(f'    ResNet-50 best val bal_acc {best:.4f}')
    model.load_state_dict(best_state)
    lv, lt = logits_of(dl_va), logits_of(dl_te)
    free(model)
    return lv, lt

## 8. Main experiment — patient-disjoint 5-fold CV, both tasks
**OURS** = `LoRA Phikon-v2 + stain aug (ours)`. Seed 0 predictions are kept out-of-fold for the paired tests.

In [9]:
OURS = 'LoRA Phikon-v2 + stain aug (ours)'
OOF = defaultdict(dict)
OOF_RAW = defaultdict(dict)
t_main = time.time()
for task in CFG['tasks']:
    y, C = df[f'y_{task}'].values, TASK_C[task]
    for f in range(CFG['n_folds']):
        tr_all = np.where(df.fold != f)[0]
        te = np.where(df.fold == f)[0]
        tr, va = inner_split(tr_all, seed=f)
        Xs = [scale(tr, X)[0] for X in FEATS]
        Xc = np.concatenate(Xs, 1)
        print(f'\n=== {task} | fold {f} | train {len(tr)} val {len(va)} test {len(te)} ({len(set(PATIENT[te]))} test patients) ===')

        def put(method, P, Praw):
            OOF[task].setdefault(method, np.zeros((N, C)))[te] = P
            OOF_RAW[task].setdefault(method, np.zeros((N, C)))[te] = Praw

        def rec_sk(method, Pt, Pv):
            put(method, record('main', task, method, f, 0, Pt, y[te], PATIENT[te], np.log(Pv + 1e-12), y[va], np.log(Pt + 1e-12)), Pt)

        # frozen baselines
        for bi, name in enumerate(BB):
            clf = logreg_fit(Xs[bi][tr], y[tr], Xs[bi][va], y[va], C)
            rec_sk(f'LinProbe {name} (frozen)', full_proba(clf, Xs[bi][te], C), full_proba(clf, Xs[bi][va], C))
        clf = logreg_fit(Xc[tr], y[tr], Xc[va], y[va], C)
        rec_sk('LogReg concat (frozen)', full_proba(clf, Xc[te], C), full_proba(clf, Xc[va], C))
        pca = PCA(min(CFG['svm_pca'], len(tr) - 1, Xc.shape[1]), random_state=0).fit(Xc[tr])
        Zp = pca.transform(Xc)
        svm = SVC(C=CFG['svm_C'], gamma='scale', probability=True, class_weight='balanced', random_state=0).fit(Zp[tr], y[tr])
        rec_sk('RBF-SVM concat (frozen)', full_proba(svm, Zp[te], C), full_proba(svm, Zp[va], C))
        knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
        rec_sk('k-NN concat (frozen)', full_proba(knn, Xc[te], C), full_proba(knn, Xc[va], C))
        mlp = train_mlp(Xc[tr], y[tr], Xc[va], y[va], C, 0)
        (lv, _), (lt, zt) = mlp_out(mlp, Xc[va]), mlp_out(mlp, Xc[te])
        put('MLP head concat (frozen)', record('main', task, 'MLP head concat (frozen)', f, 0, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt), softmax_np(lt))
        add_retrieval(task, 'raw concat (frozen)', f, Xc[te], te, y)
        for bi, name in enumerate(BB):
            add_retrieval(task, f'raw {name} (frozen)', f, Xs[bi][te], te, y)
        add_retrieval(task, 'MLP head concat (frozen)', f, zt, te, y)

        # proposed model
        for s in CFG['lora_seeds']:
            model = train_peft(tr, va, y, C, seed=s)
            lv, _ = peft_predict(model, PATHS[va])
            lt, ft = peft_predict(model, PATHS[te])
            free(model)
            P = record('main', task, OURS, f, s, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt)
            if s == CFG['lora_seeds'][0]:
                put(OURS, P, softmax_np(lt))
                add_retrieval(task, OURS, f, ft, te, y)

        if CFG['run_finetune']:
            try:
                lv, lt = finetune_resnet50(tr, va, te, y, C)
                put('ResNet-50 fine-tuned', record('main', task, 'ResNet-50 fine-tuned', f, 0, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt), softmax_np(lt))
            except Exception as e:
                print('ResNet fine-tune failed:', e)
        pd.DataFrame(ROWS).to_csv(RES_DIR / 'all_rows.csv', index=False)
print(f'\nmain experiment done in {(time.time()-t_main)/60:.1f} min')


=== binary | fold 0 | train 4909 val 1103 test 1649 (17 test patients) ===


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.48M / backbone 304.5M
    epoch 1/6: val bal_acc 0.9705  val nll 0.1417  (201s)
    epoch 2/6: val bal_acc 0.9635  val nll 0.1713  (199s)
    epoch 3/6: val bal_acc 0.9590  val nll 0.1802  (201s)
    epoch 4/6: val bal_acc 0.9648  val nll 0.1543  (200s)


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

...

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.4133  val nll 1.5816  (202s)
    epoch 2/6: val bal_acc 0.4700  val nll 1.5444  (201s)
    epoch 3/6: val bal_acc 0.4342  val nll 1.6474  (202s)
    epoch 4/6: val bal_acc 0.4310  val nll 1.5754  (201s)
    epoch 5/6: val bal_acc 0.4357  val nll 1.6247  (201s)
    ResNet-50 best val bal_acc 0.4435

=== subtype | fold 3 | train 4500 val 1468 test 1693 (16 test patients) ===


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.3485  val nll 1.5245  (212s)
    epoch 2/6: val bal_acc 0.3507  val nll 1.5929  (211s)
    epoch 3/6: val bal_acc 0.3490  val nll 1.6480  (211s)
    epoch 4/6: val bal_acc 0.3777  val nll 1.5289  (212s)
    epoch 5/6: val bal_acc 0.3660  val nll 1.5637  (212s)
    epoch 6/6: val bal_acc 0.3707  val nll 1.5479  (210s)
    ResNet-50 best val bal_acc 0.3968

=== subtype | fold 4 | train 4767 val 1494 test 1400 (17 test patients) ===


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.3722  val nll 1.4926  (221s)
    epoch 2/6: val bal_acc 0.3748  val nll 1.4051  (221s)
    epoch 3/6: val bal_acc 0.3532  val nll 1.4827  (218s)
    epoch 4/6: val bal_acc 0.3484  val nll 1.4754  (219s)
    epoch 5/6: val bal_acc 0.3743  val nll 1.4311  (220s)
    ResNet-50 best val bal_acc 0.2670

main experiment done in 313.6 min


## 9. Ablations (subtype task, seed 0)

In [10]:
ABL = {'Abl: no stain aug': dict(stain=False),
       'Abl: LoRA on DINOv2 (general FM)': dict(backbone='dinov2_vitb14')}
if CFG['abl_supcon']:
    ABL['Abl: + PD-SupCon'] = dict(supcon=True)
if CFG['run_ablation']:
    task = 'subtype' if 'subtype' in CFG['tasks'] else CFG['tasks'][0]
    y, C = df[f'y_{task}'].values, TASK_C[task]
    for name, kw in ABL.items():
        for f in range(CFG['n_folds']):
            tr_all = np.where(df.fold != f)[0]; te = np.where(df.fold == f)[0]
            tr, va = inner_split(tr_all, seed=f)
            print(f'\n--- {name} | fold {f}')
            model = train_peft(tr, va, y, C, seed=0, **kw)
            lv, _ = peft_predict(model, PATHS[va])
            lt, ft = peft_predict(model, PATHS[te])
            free(model)
            P = record('ablation', task, name, f, 0, softmax_np(lt), y[te], PATIENT[te], lv, y[va], lt)
            OOF[task].setdefault(name, np.zeros((N, C)))[te] = P
            add_retrieval(task, name, f, ft, te, y)
        pd.DataFrame(ROWS).to_csv(RES_DIR / 'all_rows.csv', index=False)


--- Abl: no stain aug | fold 0


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.4171  val nll 1.7575  (201s)
    epoch 2/6: val bal_acc 0.4059  val nll 1.8705  (201s)
    epoch 3/6: val bal_acc 0.4445  val nll 1.8102  (200s)
    epoch 4/6: val bal_acc 0.4287  val nll 1.8269  (200s)
    epoch 5/6: val bal_acc 0.4322  val nll 1.7970  (200s)
    epoch 6/6: val bal_acc 0.4276  val nll 1.7970  (200s)

--- Abl: no stain aug | fold 1


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.2168  val nll 1.7950  (202s)
    epoch 2/6: val bal_acc 0.1634  val nll 2.1378  (203s)
    epoch 3/6: val bal_acc 0.1928  val nll 1.9445  (202s)
    epoch 4/6: val bal_acc 0.1784  val nll 2.0250  (203s)

--- Abl: no stain aug | fold 2


...

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.4254  val nll 1.6021  (200s)
    epoch 2/6: val bal_acc 0.5035  val nll 1.5000  (201s)
    epoch 3/6: val bal_acc 0.4622  val nll 1.6290  (201s)
    epoch 4/6: val bal_acc 0.4404  val nll 1.5885  (200s)
    epoch 5/6: val bal_acc 0.4520  val nll 1.6223  (201s)

--- Abl: no stain aug | fold 3


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.3315  val nll 1.5652  (211s)
    epoch 2/6: val bal_acc 0.3658  val nll 1.5564  (212s)
    epoch 3/6: val bal_acc 0.3570  val nll 1.6429  (212s)
    epoch 4/6: val bal_acc 0.3820  val nll 1.5348  (212s)
    epoch 5/6: val bal_acc 0.3636  val nll 1.5642  (212s)
    epoch 6/6: val bal_acc 0.3676  val nll 1.5573  (212s)

--- Abl: no stain aug | fold 4


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    LoRA modules 72 | trainable 1.49M / backbone 304.5M
    epoch 1/6: val bal_acc 0.3527  val nll 1.5159  (220s)
    epoch 2/6: val bal_acc 0.3317  val nll 1.4866  (221s)
    epoch 3/6: val bal_acc 0.3204  val nll 1.5689  (221s)
    epoch 4/6: val bal_acc 0.3271  val nll 1.5230  (221s)

--- Abl: LoRA on DINOv2 (general FM) | fold 0
    LoRA modules 12 | trainable 0.53M / backbone 86.0M
    epoch 1/6: val bal_acc 0.4004  val nll 1.9586  (83s)
    epoch 2/6: val bal_acc 0.4467  val nll 1.6831  (82s)
    epoch 3/6: val bal_acc 0.4391  val nll 1.6822  (82s)
    epoch 4/6: val bal_acc 0.4614  val nll 1.7482  (83s)
    epoch 5/6: val bal_acc 0.4546  val nll 1.7068  (83s)
    epoch 6/6: val bal_acc 0.4533  val nll 1.7168  (84s)

--- Abl: LoRA on DINOv2 (general FM) | fold 1
    LoRA modules 12 | trainable 0.53M / backbone 86.0M
    epoch 1/6: val bal_acc 0.2289  val nll 2.0729  (84s)
    epoch 2/6: val bal_acc 0.2321  val nll 1.9463  (84s)
    epoch 3/6: val bal_acc 0.1947  val nll 1.9335  (

## 10. Leakage demonstration — identical models, image-level vs patient-level splits

In [11]:
if CFG['run_leakage_demo']:
    task = 'subtype' if 'subtype' in CFG['tasks'] else CFG['tasks'][0]
    y, C = df[f'y_{task}'].values, TASK_C[task]
    img_folds = np.zeros(N, int)
    for f, (_, te) in enumerate(StratifiedKFold(CFG['n_folds'], shuffle=True, random_state=42).split(df, y)):
        img_folds[te] = f
    for split_name, folds in [('patient-level', df.fold.values), ('image-level (leaky)', img_folds)]:
        for f in range(CFG['n_folds']):
            tr = np.where(folds != f)[0]; te = np.where(folds == f)[0]
            Xc = np.concatenate([scale(tr, X)[0] for X in FEATS], 1)
            ex = dict(split=split_name)
            clf = LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced').fit(Xc[tr], y[tr])
            record('leakage', task, 'LogReg concat (frozen)', f, 0, full_proba(clf, Xc[te], C), y[te], PATIENT[te], extra=ex)
            knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
            record('leakage', task, 'k-NN concat (frozen)', f, 0, full_proba(knn, Xc[te], C), y[te], PATIENT[te], extra=ex)
        print(f'leakage demo: {split_name} done')
    pd.DataFrame(ROWS).to_csv(RES_DIR / 'all_rows.csv', index=False)

leakage demo: patient-level done
leakage demo: image-level (leaky) done


## 11. External validation — BACH (ICIAR 2018), benign vs malignant, no retraining on BACH
The dataset must be attached with **Add Input** (see the top cell). Mapping: Normal + Benign → benign,
InSitu + Invasive → malignant. BACH has no patient IDs, so metrics are image-level.

In [12]:
EXT_LABEL = {'normal': 0, 'benign': 0, 'insitu': 1, 'in situ': 1, 'invasive': 1}


def make_synthetic_bach(root, n=8):
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    rng = np.random.default_rng(1)
    for cls, si in [('Normal', 1), ('Benign', 2), ('InSitu', 5), ('Invasive', 6)]:
        d = root / 'ICIAR2018_BACH_Challenge' / 'Photos' / cls
        d.mkdir(parents=True, exist_ok=True)
        for k in range(n):
            _synth_img(rng, si, 200, 384, 512).save(d / f'{cls[:2].lower()}{k:03d}.tif')
    return root


def scan_external(root):
    er = []
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() not in ('.tif', '.tiff', '.png', '.jpg', '.jpeg') or 'mask' in str(p).lower():
            continue
        if RX.search(p.name) or 'breakhis' in str(p).lower():      # never pick up BreakHis files
            continue
        par = [a.name for a in p.parents if a.name.lower() in EXT_LABEL]
        if not par or p.stat().st_size > 60e6:
            continue
        er.append(dict(path=str(p), y=EXT_LABEL[par[0].lower()], cls=par[0]))
    return pd.DataFrame(er, columns=['path', 'y', 'cls']).drop_duplicates('path')


EXT = None
if CFG['run_external']:
    try:
        if DRY_RUN:
            EXT = scan_external(make_synthetic_bach(WORK / 'synthetic_bach'))
        else:
            roots = [d for pat in ('*', '*/*', '*/*/*') for d in Path('/kaggle/input').glob(pat)
                     if d.is_dir() and 'bach' in d.name.lower()]
            print('attached BACH folders:', [str(r) for r in roots])
            EXT = pd.concat([scan_external(r) for r in roots]).drop_duplicates('path') if roots else pd.DataFrame()
            if len(EXT) == 0:
                import kagglehub
                EXT = scan_external(kagglehub.dataset_download(CFG['external_slug']))
        if QUICK_TEST and len(EXT):
            EXT = EXT.groupby('cls').head(20).reset_index(drop=True)
        print('external images:', len(EXT), dict(EXT.cls.value_counts()) if len(EXT) else '')
        assert len(EXT) > 0 and EXT.y.nunique() == 2
    except Exception as e:
        print(f'!! external dataset unavailable (attach it with Add Input): {type(e).__name__}: {e}')
        EXT = None

EXT_OOF = {}
if EXT is not None:
    y_ext, ext_pid = EXT.y.values, np.arange(len(EXT)).astype(str)
    y = df.y_binary.values
    tr, va = inner_split(np.arange(N), seed=0)
    Xs, Es = [], []
    for name, X in zip(BB, FEATS):
        xs, sc = scale(tr, X)
        Xs.append(xs); Es.append(sc.transform(extract_frozen(name, EXT.path.values, f'ext_{TAG}')).astype(np.float32))
    Xc, Ec = np.concatenate(Xs, 1), np.concatenate(Es, 1)

    def rec_ext(method, Pt, Pv):
        EXT_OOF[method] = record('external', 'binary', method, -1, 0, Pt, y_ext, ext_pid, np.log(Pv + 1e-12), y[va], np.log(Pt + 1e-12))

    for bi, name in enumerate(BB):
        clf = logreg_fit(Xs[bi][tr], y[tr], Xs[bi][va], y[va], 2)
        rec_ext(f'LinProbe {name} (frozen)', full_proba(clf, Es[bi], 2), full_proba(clf, Xs[bi][va], 2))
    clf = logreg_fit(Xc[tr], y[tr], Xc[va], y[va], 2)
    rec_ext('LogReg concat (frozen)', full_proba(clf, Ec, 2), full_proba(clf, Xc[va], 2))
    pca = PCA(min(CFG['svm_pca'], len(tr) - 1, Xc.shape[1]), random_state=0).fit(Xc[tr])
    svm = SVC(C=CFG['svm_C'], gamma='scale', probability=True, class_weight='balanced', random_state=0).fit(pca.transform(Xc[tr]), y[tr])
    rec_ext('RBF-SVM concat (frozen)', full_proba(svm, pca.transform(Ec), 2), full_proba(svm, pca.transform(Xc[va]), 2))
    knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[tr], y[tr])
    rec_ext('k-NN concat (frozen)', full_proba(knn, Ec, 2), full_proba(knn, Xc[va], 2))
    mlp = train_mlp(Xc[tr], y[tr], Xc[va], y[va], 2, 0)
    rec_ext('MLP head concat (frozen)', softmax_np(mlp_out(mlp, Ec)[0]), softmax_np(mlp_out(mlp, Xc[va])[0]))
    for name, kw in [(OURS, {}), ('Abl: no stain aug', dict(stain=False))]:
        print(f'\n--- external: {name}')
        model = train_peft(tr, va, y, 2, seed=0, **kw)
        lv, _ = peft_predict(model, PATHS[va])
        lt, _ = peft_predict(model, EXT.path.values)
        free(model)
        EXT_OOF[name] = record('external', 'binary', name, -1, 0, softmax_np(lt), y_ext, ext_pid, lv, y[va], lt)
    pd.DataFrame(ROWS).to_csv(RES_DIR / 'all_rows.csv', index=False)
    print('external validation done')

attached BACH folders: []
!! external dataset unavailable (attach it with Add Input): BackendError: POST failed with: {"errors":["New Datasets cannot be attached in non-interactive sessions. Found no versions attached for Dataset [truthisneverlinear/bach-breast-cancer-histology-images]."],"error":{"code":9},"wasSuccessful":false}


## 12. Results tables and significance tests

In [13]:
R = pd.DataFrame(ROWS)
R.to_csv(RES_DIR / 'all_rows.csv', index=False)
pd.set_option('display.width', 250)
KEYS = ['acc', 'bal_acc', 'macro_f1', 'auc', 'pat_acc', 'pat_bal_acc', 'prr', 'ece', 'ece_raw']


def summarize(sub, by):
    ks = [k for k in KEYS if k in sub and sub[k].notna().any()]
    agg = sub.groupby(by)[ks].agg(['mean', 'std'])
    out = pd.DataFrame(index=agg.index)
    for k in ks:
        out[k] = [f'{m*100:.2f} ± {0 if np.isnan(s) else s*100:.2f}' if not k.startswith('ece')
                  else f'{m:.4f} ± {0 if np.isnan(s) else s:.4f}' for m, s in agg[k].values]
    return out.loc[sub.groupby(by).bal_acc.mean().sort_values(ascending=False).index]


main = R[R.experiment == 'main']
for task in CFG['tasks']:
    t = summarize(main[main.task == task], 'method'); t.to_csv(RES_DIR / f'table_main_{task}.csv')
    print(f'\n=== MAIN — {task} (mean ± std over folds, %) ===\n{t.to_string()}')
if 'ablation' in set(R.experiment):
    a = pd.concat([main[(main.task == 'subtype') & (main.method == OURS)], R[R.experiment == 'ablation']])
    t = summarize(a, 'method'); t.to_csv(RES_DIR / 'table_ablation.csv')
    print(f'\n=== ABLATION — subtype ===\n{t.to_string()}')
if 'retrieval' in set(R.experiment):
    rt = R[R.experiment == 'retrieval'].pivot_table(index=['task', 'method'], values=['map_at_r', 'p_at_10'], aggfunc='mean') * 100
    rt.round(2).to_csv(RES_DIR / 'table_retrieval.csv')
    print(f'\n=== RETRIEVAL on unseen patients (%) ===\n{rt.round(2).to_string()}')
if 'leakage' in set(R.experiment):
    lk = R[R.experiment == 'leakage'].pivot_table(index='method', columns='split', values=['bal_acc', 'acc'], aggfunc='mean') * 100
    lk.round(2).to_csv(RES_DIR / 'table_leakage.csv')
    print(f'\n=== LEAKAGE DEMO (%) ===\n{lk.round(2).to_string()}')
if 'external' in set(R.experiment):
    ex = summarize(R[R.experiment == 'external'], 'method'); ex.to_csv(RES_DIR / 'table_external_bach.csv')
    print(f'\n=== EXTERNAL — BACH, trained on BreakHis only (%) ===\n{ex.to_string()}')


def bal_acc_fast(y, pred):
    return float(np.mean([(pred[y == c] == c).mean() for c in np.unique(y)]))


def cluster_bootstrap(stat_fn, pids, n_boot, seed=0):
    rng = np.random.default_rng(seed)
    up = np.unique(pids)
    groups = pd.Series(np.arange(len(pids))).groupby(pids).apply(np.array).to_dict()
    obs = stat_fn(np.arange(len(pids)))
    diffs = np.array([stat_fn(np.concatenate([groups[p] for p in rng.choice(up, len(up))])) for _ in range(n_boot)])
    return obs, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))


def verdict(lo_, hi_):
    return 'OURS better' if lo_ > 0 else ('comparator better' if hi_ < 0 else 'no significant difference')


sig = []


def cls_test(analysis, task, yv, ref_P, P, pids, name):
    ref, pb = ref_P.argmax(1), P.argmax(1)
    d, lo_, hi_, p = cluster_bootstrap(lambda ii: bal_acc_fast(yv[ii], ref[ii]) - bal_acc_fast(yv[ii], pb[ii]), pids, CFG['n_boot'])
    b = int(((ref == yv) & (pb != yv)).sum()); c = int(((ref != yv) & (pb == yv)).sum())
    sig.append(dict(analysis=analysis, task=task, comparator=name, delta_pp=d * 100, ci95_lo=lo_ * 100, ci95_hi=hi_ * 100,
                    p_bootstrap=p, mcnemar_b=b, mcnemar_c=c, p_mcnemar=binomtest(b, b + c, 0.5).pvalue if b + c else 1.0,
                    verdict=verdict(lo_, hi_)))


for task in CFG['tasks']:
    yv = df[f'y_{task}'].values
    if OURS not in OOF[task]:
        continue
    for m, P in OOF[task].items():
        if m != OURS:
            cls_test('classification', task, yv, OOF[task][OURS], P, PATIENT, m)
    A = AP_OOF[task].get(OURS)
    for m, B in AP_OOF[task].items():
        if A is None or m == OURS:
            continue
        ok = ~np.isnan(A) & ~np.isnan(B)
        if ok.sum() >= 10:
            a_, b_ = A[ok], B[ok]
            d, lo_, hi_, p = cluster_bootstrap(lambda ii: a_[ii].mean() - b_[ii].mean(), PATIENT[ok], CFG['n_boot'])
            sig.append(dict(analysis='retrieval mAP@R', task=task, comparator=m, delta_pp=d * 100, ci95_lo=lo_ * 100,
                            ci95_hi=hi_ * 100, p_bootstrap=p, verdict=verdict(lo_, hi_)))
if OURS in EXT_OOF:
    for m, P in EXT_OOF.items():
        if m != OURS:
            cls_test('external BACH', 'binary', y_ext, EXT_OOF[OURS], P, np.arange(len(y_ext)), m)
sig = pd.DataFrame(sig)
sig.to_csv(RES_DIR / 'table_significance.csv', index=False)
print('\n=== OURS vs each comparator — paired patient-cluster bootstrap 95% CI (balanced accuracy / mAP@R, pp) ===')
print(sig.round(4).to_string(index=False))


=== MAIN — binary (mean ± std over folds, %) ===
                                            acc       bal_acc      macro_f1           auc        pat_acc    pat_bal_acc           prr              ece          ece_raw
method                                                                                                                                                                 
LoRA Phikon-v2 + stain aug (ours)  91.43 ± 6.57  90.41 ± 7.50  88.92 ± 6.52  96.28 ± 4.26   92.40 ± 8.49   92.60 ± 9.04  90.75 ± 8.34  0.0497 ± 0.0349  0.0708 ± 0.0423
MLP head concat (frozen)           89.02 ± 7.62  88.94 ± 8.35  86.15 ± 6.36  94.96 ± 5.33  92.16 ± 14.48  91.73 ± 14.30  88.39 ± 9.49  0.0694 ± 0.0622  0.0708 ± 0.0281
RBF-SVM concat (frozen)            90.28 ± 8.13  88.86 ± 8.04  88.29 ± 7.24  95.88 ± 4.78   94.82 ± 8.67   94.23 ± 9.01  89.85 ± 9.62  0.0616 ± 0.0528  0.0669 ± 0.0697
LogReg concat (frozen)             89.43 ± 7.58  88.84 ± 7.82  86.87 ± 6.12  94.96 ± 5.68  93.49 ± 11.55  92.9

## 13. Claims checklist — write ONLY the SUPPORTED claims into the paper

In [14]:
def claim(text, ok, evidence):
    print(f"[{'SUPPORTED    ' if ok else 'NOT SUPPORTED'}] {text}\n      evidence: {evidence}")
    return dict(claim=text, supported=bool(ok), evidence=evidence)


def ev(s):
    return '; '.join(f'{r.comparator}: {r.delta_pp:+.2f}pp [{r.ci95_lo:.2f},{r.ci95_hi:.2f}]' for r in s.itertuples())


claims = []
for task in CFG['tasks']:
    s = sig[(sig.analysis == 'classification') & (sig.task == task) & ~sig.comparator.str.startswith('Abl')] if len(sig) else sig
    if len(s):
        claims.append(claim(f'[{task}] OURS beats EVERY baseline (patient-level CV, balanced accuracy)', (s.verdict == 'OURS better').all(), ev(s)))
        fz = s[s.comparator.str.contains('frozen')]
        claims.append(claim(f'[{task}] OURS beats every FROZEN-feature method (LoRA adaptation breaks the frozen ceiling)',
                            len(fz) > 0 and (fz.verdict == 'OURS better').all(), ev(fz)))
    s = sig[(sig.analysis == 'retrieval mAP@R') & (sig.task == task)] if len(sig) else sig
    if len(s):
        claims.append(claim(f'[{task}] OURS embedding retrieves better than every comparator', (s.verdict == 'OURS better').all(), ev(s)))
if len(sig):
    for comp, txt in [('Abl: no stain aug', 'stain augmentation improves subtype accuracy'),
                      ('Abl: LoRA on DINOv2 (general FM)', 'a pathology FM is a better base than a general FM under LoRA'),
                      ('Abl: + PD-SupCon', 'plain CE is at least as good as adding PD-SupCon (OURS not worse)')]:
        r = sig[(sig.analysis == 'classification') & (sig.comparator == comp)]
        if len(r):
            r = r.iloc[0]
            ok = (r.verdict == 'OURS better') if 'not worse' not in txt else (r.verdict != 'comparator better')
            claims.append(claim(f'[subtype] {txt}', ok, f'{r.delta_pp:+.2f}pp [{r.ci95_lo:.2f},{r.ci95_hi:.2f}]'))
    s = sig[(sig.analysis == 'external BACH') & ~sig.comparator.str.startswith('Abl')]
    if len(s):
        claims.append(claim('[external BACH] OURS generalises better than EVERY baseline', (s.verdict == 'OURS better').all(), ev(s)))
    r = sig[(sig.analysis == 'external BACH') & (sig.comparator == 'Abl: no stain aug')]
    if len(r):
        r = r.iloc[0]
        claims.append(claim('[external BACH] stain augmentation improves cross-site generalisation', r.verdict == 'OURS better',
                            f'{r.delta_pp:+.2f}pp [{r.ci95_lo:.2f},{r.ci95_hi:.2f}]'))
if 'leakage' in set(R.experiment):
    lk = R[R.experiment == 'leakage'].groupby('split').bal_acc.mean() * 100
    claims.append(claim('Image-level splitting inflates BreakHis subtype accuracy by >20 pp',
                        lk.get('image-level (leaky)', 0) - lk.get('patient-level', 0) > 20, lk.round(2).to_dict()))
if len(audit):
    claims.append(claim('BreakHis contains cross-label duplicate images', (audit.kind == 'cross_label').any(),
                        audit.groupby(['kind', 'patients', 'subtypes']).size().to_dict()))
pd.DataFrame(claims).to_csv(RES_DIR / 'claims_checklist.csv', index=False)

[NOT SUPPORTED] [binary] OURS beats EVERY baseline (patient-level CV, balanced accuracy)
      evidence: LinProbe dinov2_vitb14 (frozen): +6.21pp [2.78,10.17]; LinProbe phikon_v2 (frozen): +2.26pp [0.18,4.32]; LogReg concat (frozen): +2.31pp [-0.26,5.18]; RBF-SVM concat (frozen): +2.06pp [-0.32,4.98]; k-NN concat (frozen): +2.01pp [-0.32,4.20]; MLP head concat (frozen): +2.16pp [0.01,4.86]; ResNet-50 fine-tuned: +6.68pp [2.90,11.01]
[NOT SUPPORTED] [binary] OURS beats every FROZEN-feature method (LoRA adaptation breaks the frozen ceiling)
      evidence: LinProbe dinov2_vitb14 (frozen): +6.21pp [2.78,10.17]; LinProbe phikon_v2 (frozen): +2.26pp [0.18,4.32]; LogReg concat (frozen): +2.31pp [-0.26,5.18]; RBF-SVM concat (frozen): +2.06pp [-0.32,4.98]; k-NN concat (frozen): +2.01pp [-0.32,4.20]; MLP head concat (frozen): +2.16pp [0.01,4.86]
[NOT SUPPORTED] [binary] OURS embedding retrieves better than every comparator
      evidence: raw concat (frozen): +7.18pp [5.43,8.88]; raw dinov2_vit

## 14. Figures (PNG for review, PDF for camera-ready)

In [15]:
OKABE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442', '#000000']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig); print('saved', name)


def hbar(s, name, xlabel):
    s = s.sort_values('mean')
    fig, ax = plt.subplots(figsize=(7.8, 0.42 * len(s) + 1.2))
    ax.barh(s.index, s['mean'] * 100, xerr=s['std'].fillna(0) * 100, capsize=2,
            color=[OKABE[3] if i == OURS else (OKABE[5] if str(i).startswith('Abl') else OKABE[0]) for i in s.index])
    ax.set_xlabel(xlabel); ax.set_xlim(max(0, s['mean'].min() * 100 - 10), min(100, s['mean'].max() * 100 + 6))
    save(fig, name)


for task in CFG['tasks']:
    sub = main[main.task == task]
    if len(sub):
        hbar(sub.groupby('method').bal_acc.agg(['mean', 'std']), f'F1_main_{task}', f'Balanced accuracy (%) — {task}, patient-disjoint CV')
        hbar(sub.groupby('method').pat_bal_acc.agg(['mean', 'std']), f'F2_patient_{task}', f'Patient-level balanced accuracy (%) — {task}')
    rr = R[(R.experiment == 'retrieval') & (R.task == task)]
    if len(rr):
        hbar(rr.groupby('method').map_at_r.agg(['mean', 'std']), f'F3_retrieval_{task}', f'Cross-patient retrieval mAP@R (%) — {task}')
if 'ablation' in set(R.experiment):
    a = pd.concat([main[(main.task == 'subtype') & (main.method == OURS)], R[R.experiment == 'ablation']])
    hbar(a.groupby('method').bal_acc.agg(['mean', 'std']), 'F4_ablation', 'Balanced accuracy (%) — subtype')
if 'external' in set(R.experiment):
    hbar(R[R.experiment == 'external'].groupby('method').bal_acc.agg(['mean', 'std']), 'F5_external_bach',
         'Balanced accuracy (%) on BACH — trained on BreakHis only')
if 'leakage' in set(R.experiment):
    lk = R[R.experiment == 'leakage'].groupby(['method', 'split']).bal_acc.mean().unstack() * 100
    fig, ax = plt.subplots(figsize=(6, 3.5)); x = np.arange(len(lk))
    for i, c_ in enumerate(lk.columns):
        ax.bar(x + (i - 0.5) * 0.38, lk[c_], 0.38, label=c_, color=OKABE[i])
    ax.set_xticks(x); ax.set_xticklabels(lk.index); ax.set_ylabel('Balanced accuracy (%)'); ax.set_ylim(0, 100)
    ax.legend(frameon=False); save(fig, 'F6_leakage')

task = 'subtype' if 'subtype' in CFG['tasks'] else CFG['tasks'][0]
y = df[f'y_{task}'].values
if OURS in OOF[task]:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    for lab, P, col in [('Before T-scaling', OOF_RAW[task][OURS], OKABE[1]), ('After T-scaling', OOF[task][OURS], OKABE[0])]:
        conf, acc_ = P.max(1), (P.argmax(1) == y)
        xs, ys = [], []
        for lo_, hi_ in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):
            m = (conf > lo_) & (conf <= hi_)
            if m.sum() >= 5:
                xs.append(conf[m].mean()); ys.append(acc_[m].mean())
        ax.plot(xs, ys, 'o-', color=col, label=f'{lab} (ECE {ece_score(P, y):.3f})')
    ax.plot([0, 1], [0, 1], '--', color='grey'); ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.legend(frameon=False, fontsize=8); save(fig, 'F7_reliability')

    P = OOF[task][OURS]
    Pp, yp, _ = patient_agg(P, y, PATIENT)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, (yt, pr, ttl) in zip(axes, [(y, P.argmax(1), 'Image level'), (yp, Pp.argmax(1), 'Patient level')]):
        cm = confusion_matrix(yt, pr, labels=range(TASK_C[task]))
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        names = TASK_NAMES[task]
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
        for i in range(len(names)):
            for j in range(len(names)):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='w' if cmn[i, j] > 0.5 else 'k')
        ax.set_title(f'OURS — {ttl}'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    save(fig, 'F8_confusion')

saved F1_main_binary
saved F2_patient_binary
saved F3_retrieval_binary
saved F1_main_subtype
saved F2_patient_subtype
saved F3_retrieval_subtype
saved F4_ablation
saved F6_leakage
saved F7_reliability
saved F8_confusion


## 15. Export

In [16]:
shutil.make_archive(str(WORK / 'cmfe_v3_results'), 'zip', root_dir=WORK, base_dir='results')
shutil.make_archive(str(WORK / 'cmfe_v3_figures'), 'zip', root_dir=WORK, base_dir='figures')
print('Done. Download cmfe_v3_results.zip and cmfe_v3_figures.zip from the Output tab. rows:', len(ROWS))

Done. Download cmfe_v3_results.zip and cmfe_v3_figures.zip from the Output tab. rows: 170


## What to read when it finishes
1. **`claims_checklist.csv`** sets the story. Write only the SUPPORTED lines.
2. The key claim is *"LoRA adaptation breaks the frozen-feature ceiling"*: OURS vs every frozen method,
   with bootstrap CIs.
3. **`table_external_bach.csv`** is the strongest evidence, because BACH was never used for any
   design choice.
4. If OURS does **not** win, the paper is still a strong benchmark: leakage collapse, label conflict,
   frozen vs adapted foundation models, calibration, external validation.